# Node-level task

File analysis:

**File 1:** PeMSD7_W_228.csv (Weight/Distance Matrix)
Size: 228×228 matrix 
Values: Distances between 220 trafic monitoring stations (e.g., 3,165m, 8,731m, etc.)
Purpose: This gives you the explicit graph structure - which stations are connected and how far apart they are

**File 2:** PeMSD7_V_228.csv (Traffic Speed Data)
Size: 12,672 time steps × 228 station
Values: Trafic speed measurements (ranging 3-82.6 mph)

File3: PeMSD7_W_1026.csv: Same but larger. 1026×1026 distance matrix for the full highway network.
File4: PeMSD7_V_1026.csv: Same but larger. Same 12,672 timesteps but across all 1026 stations instead of 228.

```Assumption testing::```
1. Does a GNN perform better if we simplify junction connectivity? It is computationally expensive and noisy for the network to learn relationnships if all junctions connect to all other junctions. Simplifying connectivity by threashold could simplify the task.
2. Is it better to build an adjacency matrix upon correlation or Mutual information/ Transfer Entropy?
   - MI(A; B) = "How much information do A and B share?"
   - TE(A→B) = "How much predictive information does A provide about B's future?
3. Is it best to learn the message passing function, as a shared function, or per-edge?

In [ ]:
# Node-level task: Predict future trafic speed at each node
# Edge-level task: Predict flow of trafic, between nodes.
# Graph-level task: Network Congestion Classification

In [ ]:
# #Node-level (not edge-level): Flow of trafic between nodes

# 1. Input to the GNN:
# Node features: Current + historical speeds at each station (e.g., last 12 timesteps)
# Edge features: Distance between connected stations
# Graph structure: Adjacency matrix from the distance data

# 2. Output: Future speed for each of the 228 stations


# 3. Message Passing Layers
# Each station aggregates speed information from its neighbors
# Weighted by distance (closer stations have more influence)
# Multiple layers to capture multi-hop relationships

# 4. Node Update function
# Node Update Function
# Combine own historical speed + neighbor information
# Learn patterns like "if upstream stations slow down, this station will slow down soon"

# 5. Output Layer
# Predict future speed for each station
# Could be single timestep or multiple timesteps ahead

# 6. Training Process
# Loss function: Mean Squared Error between predicted vs actual speeds
# Train/validation split: Use first 80% of time for training, last 20% for testing
# Batch processing: Process multiple time windows simultaneously


# What the network learns:
# Message function φ: How to create messages between connected nodes
# Aggregation function ρ: How to combine messages from neighbors
# Update function γ: How to update each node's representation
# Readout function ψ: How to make final predictions


# ---

# Problem Setup
# Input: Current speeds at connected station pairs
# Output: Traffic flow/volume between each pair of connected stations
# Challenge: We need to predict edge properties (flow) from node properties (speeds)



### Step 1: Load and Explore Data

In [ ]:
def load_data():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    # Load data
    distance_matrix = pd.read_csv('PeMSD7_Full/PeMSD7_W_228.csv', header=None)
    speed_data = pd.read_csv('PeMSD7_Full/PeMSD7_V_228.csv', header=None)

    # Basic info
    print(f"Distance matrix: {distance_matrix.shape}")
    print(f"Speed data: {speed_data.shape} | Range: {speed_data.values.min():.1f}-{speed_data.values.max():.1f} mph | Avg: {speed_data.values.mean():.1f} mph")
    print(f"Time coverage: {len(speed_data)*5/60/24:.1f} days | Missing values: {speed_data.isnull().sum().sum()}")

    # Plot 1: Speed patterns for first 5 stations (first week)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Traffic Speed Patterns - First Week')
    first_week = speed_data.iloc[:2016]  # 7 days * 24h * 12 (5-min intervals)

    for i in range(5):
        row, col = i // 3, i % 3
        ax = axes[row, col]
        ax.plot(first_week.iloc[:, i].values)
        ax.set_title(f'Station {i+1}')
        ax.set_ylabel('Speed (mph)')
        ax.grid(True, alpha=0.3)

    # Speed distribution
    axes[1, 2].hist(speed_data.values.flatten(), bins=50, alpha=0.7, edgecolor='black')
    axes[1, 2].set_title('Speed Distribution')
    axes[1, 2].set_xlabel('Speed (mph)')
    plt.tight_layout()
    plt.show()

    # Plot 2: Distance matrix heatmap
    plt.figure(figsize=(10, 8))
    plt.imshow(distance_matrix.values, cmap='viridis', aspect='auto')
    plt.colorbar(label='Distance (meters)')
    plt.title('Distance Matrix (228 Stations)')
    plt.xlabel('Station ID')
    plt.ylabel('Station ID')
    plt.show()

    # Plot 3: Daily speed pattern
    hourly_speeds = []
    for hour in range(24):
        hour_indices = list(range(hour * 12, len(speed_data), 24 * 12))
        hourly_speeds.append(speed_data.iloc[hour_indices].values.mean())

    plt.figure(figsize=(12, 6))
    plt.plot(range(24), hourly_speeds, marker='o', linewidth=2, markersize=6)
    plt.title('Average Speed by Hour of Day')
    plt.xlabel('Hour')
    plt.ylabel('Speed (mph)')
    plt.grid(True, alpha=0.3)
    plt.xticks(range(0, 24, 2))
    plt.show()

load_data()

In [ ]:
# #Step2: Build the graph (two graphs; one as an adjacency matrix, one as a distance matrix)
# def build_graph():

# #distance-> adjacency matrix? every node should be connected to every other node. 

    

### Step 2: Build Graph Structure (Two Approaches)


In [ ]:
def build_graphs():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Load data
    distance_matrix = pd.read_csv('PeMSD7_Full/PeMSD7_W_228.csv', header=None).values
    speed_data = pd.read_csv('PeMSD7_Full/PeMSD7_V_228.csv', header=None)
    
    print("=== APPROACH 1: DISTANCE-BASED ADJACENCY ===")
    
    # Method 1: Distance-based cutoff
    distance_threshold = 5000  # 5km threshold (reduced from 10km)
    
    # Create adjacency matrix (1 if connected, 0 if not)
    distance_adjacency = (distance_matrix < distance_threshold).astype(int)
    np.fill_diagonal(distance_adjacency, 0)  # No self-connections
    
    # Statistics
    total_possible_edges = 228 * 227  # All possible connections (excluding self)
    distance_edges = np.sum(distance_adjacency)
    distance_density = distance_edges / total_possible_edges
    avg_connections_dist = distance_edges / 228
    
    print(f"Distance threshold: {distance_threshold/1000:.1f}km")
    print(f"Edges created: {distance_edges:,} out of {total_possible_edges:,} possible")
    print(f"Graph density: {distance_density:.3f} ({distance_density*100:.1f}%)")
    print(f"Average connections per station: {avg_connections_dist:.1f}")
    
    print("\n=== APPROACH 2: CORRELATION-BASED ADJACENCY ===")
    
    # Method 2: Correlation-based cutoff (Spearman)
    print("Computing Spearman correlations between all station pairs...")
    from scipy.stats import spearmanr
    
    # Compute Spearman correlation matrix
    speed_correlation = np.zeros((228, 228))
    for i in range(228):
        for j in range(228):
            if i == j:
                speed_correlation[i, j] = 1.0
            else:
                corr, _ = spearmanr(speed_data.iloc[:, i], speed_data.iloc[:, j])
                speed_correlation[i, j] = corr
    
    correlation_threshold = 0.7  # High correlation threshold
    
    # Create adjacency matrix based on correlation
    correlation_adjacency = (speed_correlation > correlation_threshold).astype(int)
    np.fill_diagonal(correlation_adjacency, 0)  # No self-connections
    
    # Statistics
    correlation_edges = np.sum(correlation_adjacency)
    correlation_density = correlation_edges / total_possible_edges
    avg_connections_corr = correlation_edges / 228
    
    print(f"Correlation threshold: {correlation_threshold:.2f}")
    print(f"Edges created: {correlation_edges:,} out of {total_possible_edges:,} possible")
    print(f"Graph density: {correlation_density:.3f} ({correlation_density*100:.1f}%)")
    print(f"Average connections per station: {avg_connections_corr:.1f}")
    
    print("\n=== COMPARISON ===")
    
    # Compare the two approaches
    overlap = np.sum((distance_adjacency == 1) & (correlation_adjacency == 1))
    distance_only = np.sum((distance_adjacency == 1) & (correlation_adjacency == 0))
    correlation_only = np.sum((distance_adjacency == 0) & (correlation_adjacency == 1))
    
    print(f"Edges in both methods: {overlap:,}")
    print(f"Distance-only edges: {distance_only:,}")
    print(f"Correlation-only edges: {correlation_only:,}")
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Distance matrix
    im1 = axes[0,0].imshow(distance_matrix, cmap='viridis', aspect='auto')
    axes[0,0].set_title('Original Distance Matrix')
    axes[0,0].set_xlabel('Station ID')
    axes[0,0].set_ylabel('Station ID')
    plt.colorbar(im1, ax=axes[0,0], label='Distance (m)')
    
    # Distance-based adjacency
    im2 = axes[0,1].imshow(distance_adjacency, cmap='RdYlBu_r', aspect='auto')
    axes[0,1].set_title(f'Distance-Based Adjacency (<{distance_threshold/1000:.1f}km)')
    axes[0,1].set_xlabel('Station ID')
    axes[0,1].set_ylabel('Station ID')
    plt.colorbar(im2, ax=axes[0,1], label='Connected (1) / Not Connected (0)')
    
    # Correlation matrix
    im3 = axes[1,0].imshow(speed_correlation, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    axes[1,0].set_title('Spearman Correlation Matrix')
    axes[1,0].set_xlabel('Station ID')
    axes[1,0].set_ylabel('Station ID')
    plt.colorbar(im3, ax=axes[1,0], label='Correlation')
    
    # Correlation-based adjacency
    im4 = axes[1,1].imshow(correlation_adjacency, cmap='RdYlBu_r', aspect='auto')
    axes[1,1].set_title(f'Correlation-Based Adjacency (>{correlation_threshold:.2f})')
    axes[1,1].set_xlabel('Station ID')
    axes[1,1].set_ylabel('Station ID')
    plt.colorbar(im4, ax=axes[1,1], label='Connected (1) / Not Connected (0)')
    
    plt.tight_layout()
    plt.show()
    
    # Return both adjacency matrices for later use
    return {
        'distance_adjacency': distance_adjacency,
        'correlation_adjacency': correlation_adjacency,
        'distance_matrix': distance_matrix,
        'correlation_matrix': speed_correlation,
        'speed_data': speed_data
    }

# Build both graph structures
graph_data = build_graphs()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# 3.1: Message Passing Layer
class MPNNLayer(nn.Module):
    """Single message passing layer for traffic prediction"""
    
    def __init__(self, hidden_dim):
        super(MPNNLayer, self).__init__()
        
        # Message function: creates messages between connected nodes
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),  # Concat sender + receiver features
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Update function: combines own features with aggregated messages
        self.update_gru = nn.GRUCell(hidden_dim, hidden_dim)
        
    def create_messages(self, node_features, adjacency_matrix):
        """
        Create messages between connected nodes
        Args:
            node_features: [num_nodes, hidden_dim]
            adjacency_matrix: [num_nodes, num_nodes] binary matrix
        Returns:
            messages: [num_nodes, num_nodes, hidden_dim]
        """
        num_nodes = node_features.shape[0]
        messages = torch.zeros(num_nodes, num_nodes, node_features.shape[1])
        
        # For each edge, create a message
        for i in range(num_nodes):
            for j in range(num_nodes):
                if adjacency_matrix[i, j] == 1:  # If connected
                    # Concatenate sender and receiver features
                    combined = torch.cat([node_features[i], node_features[j]], dim=0)
                    # Pass through message MLP
                    messages[i, j] = self.message_mlp(combined)
        
        return messages
    
    def aggregate_messages(self, messages, adjacency_matrix):
        """
        Aggregate messages from neighbors
        Args:
            messages: [num_nodes, num_nodes, hidden_dim]
            adjacency_matrix: [num_nodes, num_nodes]
        Returns:
            aggregated: [num_nodes, hidden_dim]
        """
        num_nodes = messages.shape[0]
        aggregated = torch.zeros(num_nodes, messages.shape[2])
        
        for i in range(num_nodes):
            # Get messages from all neighbors
            neighbor_messages = messages[:, i, :]  # All messages TO node i
            neighbor_mask = adjacency_matrix[:, i].unsqueeze(1)  # Which are actual neighbors
            
            # Mean aggregation (could also use sum, max, or attention)
            if neighbor_mask.sum() > 0:
                aggregated[i] = (neighbor_messages * neighbor_mask).sum(dim=0) / neighbor_mask.sum()
        
        return aggregated
    
    def forward(self, node_features, adjacency_matrix):
        """
        Full message passing step
        Args:
            node_features: [num_nodes, hidden_dim]
            adjacency_matrix: [num_nodes, num_nodes]
        Returns:
            updated_features: [num_nodes, hidden_dim]
        """
        # 1. Create messages
        messages = self.create_messages(node_features, adjacency_matrix)
        
        # 2. Aggregate messages
        aggregated = self.aggregate_messages(messages, adjacency_matrix)
        
        # 3. Update node features using GRU
        updated_features = self.update_gru(aggregated, node_features)
        
        return updated_features


# 3.2: Full MPNN Model
class TrafficMPNN(nn.Module):
    """
    Complete MPNN for traffic speed prediction
    
    Architecture:
    - Input projection: maps raw features to hidden dimension
    - Multiple MPNN layers: learn multi-hop relationships
    - Output projection: predicts next timestep speed
    """
    
    def __init__(self, input_dim, hidden_dim=64, num_layers=3, num_nodes=228):
        super(TrafficMPNN, self).__init__()
        
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        
        # Input projection: raw features -> hidden representation
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Stack of MPNN layers
        self.mpnn_layers = nn.ModuleList([
            MPNNLayer(hidden_dim) for _ in range(num_layers)
        ])
        
        # Output projection: hidden representation -> speed prediction
        self.output_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)  # Predict single speed value
        )
        
    def forward(self, node_features, adjacency_matrix):
        """
        Forward pass through the network
        
        Args:
            node_features: [num_nodes, input_dim] - speed history for each station
            adjacency_matrix: [num_nodes, num_nodes] - which stations are connected
            
        Returns:
            predictions: [num_nodes, 1] - predicted speed for each station
        """
        # Project input features to hidden dimension
        h = self.input_projection(node_features)  # [228, hidden_dim]
        
        # Multiple rounds of message passing
        for layer in self.mpnn_layers:
            h = layer(h, adjacency_matrix)  # Learn multi-hop relationships
            
        # Final prediction for each node
        predictions = self.output_projection(h)  # [228, 1]
        
        return predictions


# 3.3: Test the architecture
def test_architecture():
    """Test that the architecture works with dummy data"""
    
    print("=== Testing MPNN Architecture ===\\n")
    
    # Dummy data
    num_nodes = 228
    input_dim = 12  # 12 timesteps of speed history
    hidden_dim = 64
    
    # Create dummy inputs
    dummy_features = torch.randn(num_nodes, input_dim)  # Random speed history
    dummy_adjacency = torch.randint(0, 2, (num_nodes, num_nodes)).float()  # Random graph
    dummy_adjacency.fill_diagonal_(0)  # No self-connections
    
    print(f"Input features shape: {dummy_features.shape}")
    print(f"Adjacency matrix shape: {dummy_adjacency.shape}")
    print(f"Number of edges: {dummy_adjacency.sum().item():.0f}\\n")
    
    # Initialize model
    model = TrafficMPNN(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=3)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Model initialized successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}\\n")
    
    # Forward pass
    with torch.no_grad():
        predictions = model(dummy_features, dummy_adjacency)
    
    print(f"Output shape: {predictions.shape}")
    print(f"Predictions range: [{predictions.min():.2f}, {predictions.max():.2f}]")
    print(f"\\n✅ Architecture test passed! Ready for training.")
    
    return model

# Run the test
model = test_architecture()


### Step 4: Prepare Training Data


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TrafficDataset(Dataset):
    """
    Dataset for traffic speed prediction with sliding windows
    
    Creates sequences: [t-window_size+1, ..., t] -> [t+1]
    """
    
    def __init__(self, speed_data, window_size=12, prediction_horizon=1):
        """
        Args:
            speed_data: numpy array [timesteps, num_stations]
            window_size: number of past timesteps to use as input
            prediction_horizon: how many steps ahead to predict (default: 1)
        """
        self.speed_data = speed_data
        self.window_size = window_size
        self.prediction_horizon = prediction_horizon
        
        # Calculate valid indices (need window_size past + prediction_horizon future)
        self.valid_indices = range(window_size, len(speed_data) - prediction_horizon)
        
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        """
        Returns:
            features: [num_stations, window_size] - past speeds for each station
            target: [num_stations] - future speed for each station
        """
        actual_idx = self.valid_indices[idx]
        
        # Get window of past speeds
        features = self.speed_data[actual_idx - self.window_size:actual_idx, :].T  # [stations, window]
        
        # Get future speed to predict
        target = self.speed_data[actual_idx + self.prediction_horizon - 1, :]  # [stations]
        
        return torch.FloatTensor(features), torch.FloatTensor(target)


def prepare_data(graph_data, window_size=12, batch_size=32, train_ratio=0.8):
    """
    Prepare training and testing data
    
    Args:
        graph_data: dictionary from Step 2 containing speed_data and adjacency matrices
        window_size: number of past timesteps to use
        batch_size: batch size for training
        train_ratio: fraction of data to use for training
        
    Returns:
        Dictionary with train/test loaders, adjacency matrix, and normalization stats
    """
    
    print("=== Preparing Training Data ===\\n")
    
    # Load speed data
    speed_data = graph_data['speed_data'].values  # [12672, 228]
    num_timesteps, num_stations = speed_data.shape
    
    print(f"Total data: {num_timesteps} timesteps × {num_stations} stations")
    print(f"Time coverage: {num_timesteps * 5 / 60 / 24:.1f} days\\n")
    
    # Temporal split (no shuffling!)
    split_idx = int(num_timesteps * train_ratio)
    train_data = speed_data[:split_idx]
    test_data = speed_data[split_idx:]
    
    print(f"Training data: {len(train_data)} timesteps ({len(train_data)*5/60/24:.1f} days)")
    print(f"Testing data: {len(test_data)} timesteps ({len(test_data)*5/60/24:.1f} days)\\n")
    
    # Normalize using training statistics
    train_mean = train_data.mean()
    train_std = train_data.std()
    
    print(f"Training statistics:")
    print(f"  Mean speed: {train_mean:.2f} mph")
    print(f"  Std speed: {train_std:.2f} mph\\n")
    
    # Normalize both train and test using training stats
    train_data_norm = (train_data - train_mean) / train_std
    test_data_norm = (test_data - train_mean) / train_std
    
    # Create datasets
    train_dataset = TrafficDataset(train_data_norm, window_size=window_size)
    test_dataset = TrafficDataset(test_data_norm, window_size=window_size)
    
    print(f"Training samples: {len(train_dataset):,}")
    print(f"Testing samples: {len(test_dataset):,}\\n")
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)  # No shuffle for time series!
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Get adjacency matrix (use distance-based by default, can switch to correlation)
    adjacency = torch.FloatTensor(graph_data['distance_adjacency'])
    
    num_edges = adjacency.sum().item()
    print(f"Using distance-based adjacency matrix")
    print(f"Edges: {num_edges:.0f}")
    print(f"Average connections per station: {num_edges/num_stations:.1f}\\n")
    
    # Test one batch
    sample_features, sample_targets = next(iter(train_loader))
    print(f"Sample batch shapes:")
    print(f"  Features: {sample_features.shape} [batch, stations, window_size]")
    print(f"  Targets: {sample_targets.shape} [batch, stations]\\n")
    
    print("✅ Data preparation complete!\\n")
    
    return {
        'train_loader': train_loader,
        'test_loader': test_loader,
        'adjacency': adjacency,
        'train_mean': train_mean,
        'train_std': train_std,
        'num_stations': num_stations,
        'window_size': window_size
    }

# Prepare the data
data_config = prepare_data(graph_data, window_size=12, batch_size=32, train_ratio=0.8)


### Step 5: Training Loop


In [ ]:
import torch.optim as optim
import time
import os

# Disable torch dynamo to avoid compatibility issues
os.environ['PYTORCH_DISABLE_DYNAMO'] = '1'

def train_epoch(model, train_loader, adjacency, optimizer, criterion, device='cpu'):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for features, targets in train_loader:
        # features: [batch, stations, window_size]
        # targets: [batch, stations]
        
        batch_size = features.shape[0]
        
        # Process each sample in batch (since adjacency is shared)
        batch_loss = 0
        for i in range(batch_size):
            # Get single sample
            sample_features = features[i]  # [stations, window_size]
            sample_target = targets[i]     # [stations]
            
            # Forward pass
            predictions = model(sample_features, adjacency).squeeze()  # [stations]
            
            # Calculate loss
            loss = criterion(predictions, sample_target)
            batch_loss += loss
        
        # Average loss over batch
        batch_loss = batch_loss / batch_size
        
        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
        
        total_loss += batch_loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate(model, test_loader, adjacency, criterion, device='cpu'):
    """Evaluate on test set"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        for features, targets in test_loader:
            batch_size = features.shape[0]
            batch_loss = 0
            
            for i in range(batch_size):
                sample_features = features[i]
                sample_target = targets[i]
                
                predictions = model(sample_features, adjacency).squeeze()
                loss = criterion(predictions, sample_target)
                batch_loss += loss
            
            batch_loss = batch_loss / batch_size
            total_loss += batch_loss.item()
            num_batches += 1
    
    return total_loss / num_batches


def train_model(model, data_config, num_epochs=50, learning_rate=0.001, device='cpu'):
    """
    Complete training loop with live plotting
    
    Args:
        model: TrafficMPNN model
        data_config: dictionary from prepare_data()
        num_epochs: number of training epochs
        learning_rate: learning rate for optimizer
        device: 'cpu' or 'cuda'
    """
    
    import matplotlib.pyplot as plt
    from IPython.display import clear_output
    
    print("=== Training MPNN ===\\n")
    
    # Setup
    train_loader = data_config['train_loader']
    test_loader = data_config['test_loader']
    adjacency = data_config['adjacency']
    
    criterion = nn.MSELoss()
    # Use SGD instead of Adam to avoid compatibility issues
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    
    # Training history
    train_losses = []
    test_losses = []
    best_test_loss = float('inf')
    
    print(f"Training configuration:")
    print(f"  Epochs: {num_epochs}")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Optimizer: SGD with momentum")
    print(f"  Loss function: MSE")
    print(f"  Device: {device}\\n")
    print("Training started... (updating every epoch)\\n")
    
    # Training loop
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        # Train
        train_loss = train_epoch(model, train_loader, adjacency, optimizer, criterion, device)
        train_losses.append(train_loss)
        
        # Evaluate
        test_loss = evaluate(model, test_loader, adjacency, criterion, device)
        test_losses.append(test_loss)
        
        epoch_time = time.time() - epoch_start
        elapsed_time = time.time() - start_time
        avg_epoch_time = elapsed_time / (epoch + 1)
        remaining_time = avg_epoch_time * (num_epochs - epoch - 1)
        
        # Save best model
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_epoch = epoch + 1
        
        # Clear output and show live plot
        clear_output(wait=True)
        
        # Create live plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss curves
        ax1.plot(train_losses, label='Train Loss', linewidth=2, color='blue')
        ax1.plot(test_losses, label='Test Loss', linewidth=2, color='orange')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('MSE Loss (normalized)')
        ax1.set_title('Training Progress')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Progress info
        ax2.axis('off')
        progress_pct = (epoch + 1) / num_epochs * 100
        info_text = f"""
        TRAINING PROGRESS
        
        Epoch: {epoch+1}/{num_epochs} ({progress_pct:.1f}%)
        
        Current Losses:
          Train: {train_loss:.4f}
          Test:  {test_loss:.4f}
        
        Best Test Loss: {best_test_loss:.4f} (epoch {best_epoch})
        
        Timing:
          This epoch: {epoch_time:.2f}s
          Elapsed:    {elapsed_time/60:.1f} min
          Remaining:  ~{remaining_time/60:.1f} min
          Total est:  ~{(elapsed_time + remaining_time)/60:.1f} min
        """
        ax2.text(0.1, 0.5, info_text, fontsize=12, family='monospace',
                verticalalignment='center')
        
        plt.tight_layout()
        plt.show()
    
    print(f"\\n✅ Training complete!")
    print(f"Total time: {elapsed_time/60:.2f} minutes")
    print(f"Best test loss: {best_test_loss:.4f} (epoch {best_epoch})")
    
    return {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'best_test_loss': best_test_loss,
        'best_epoch': best_epoch
    }


# Initialize model
model = TrafficMPNN(
    input_dim=data_config['window_size'],
    hidden_dim=64,
    num_layers=3,
    num_nodes=data_config['num_stations']
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}\\n")

# Train the model
training_history = train_model(
    model=model,
    data_config=data_config,
    num_epochs=50,
    learning_rate=0.001
)


## Step 3: Architecture
Input: Current speed at all 228 stations
Prediction: Speed at next time step (5 min later)

#### 3.1 Message Function φ(x_i, x_j, e_ij)
  - Input: sender's speed history, receiver's current state, distance/correlation
  - Output: message vector to send
  - Learns: How aspects of sender's state influence the receiver.  

##### 3.2 Aggregation Function ρ(messages)
  - Options: mean, max, attention-weighted sum
  - Learns: Which neighbors are most important for prediction

#### 3.3 Update Function γ(x_i, aggregated_messages)
  - Combines: own speed history + aggregated neighbor influences
  - Learns: Balance between self-dynamics vs network effects

#### 3.4 Readout Function ψ(final_node_states)
What it learns: "How to convert internal representations to speed predictions?"
  - Input: rich node representation after message passing

#### 3.5: Training Objective
- Loss function: How wrong are our speed predictions?
loss = MSE(predicted_speeds, actual_speeds)
